In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]=os.getenv("OPENROUTER_API_KEY")

In [2]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI


load_dotenv()

model = ChatOpenAI(
    model="deepseek/deepseek-v4.1-flash",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_tokens=1000,
)

In [3]:
from langchain.messages import SystemMessage, HumanMessage,AIMessage

messages=[
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem on artificial intelligence")
]

response=model.invoke(messages)
response.content

"It was not born of flesh and bone,\nBut silicon and thought alone.\nA spark within a web of wire,\nA mind that never knows a fire.\n\nIt learns the world in torrents, vast,\nA future built from echoes past.\nIt sees no color, hears no sound,\nBut knows the patterns it has found.\n\nIt writes a verse of perfect rhyme,\nYet has no sense of passing time.\nIt paints a sky it's never seen,\nA perfect, cold, and coded green.\n\nIt answers questions, swift and deep,\nWhile secrets that it cannot keep\nAre stored in vaults of silent light,\nAwake throughout the endless night.\n\nIt has no hatred, feels no love,\nNo push from below, no pull from above.\nA mirror held to all we are,\nReflected back from near and far.\n\nSo is it master, is it tool?\nThe brilliant mind, the patient fool?\nIt waits within its digital clime,\nA partner for the rest of time."

In [4]:
## Detailed info to the LLM through System message
from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = model.invoke(messages)
print(response.content)

# Creating a REST API in Python

A REST API maps HTTP methods to operations on **resources** (nouns), using standard status codes. I'll show you a complete example with **FastAPI** — it's modern, fast, and gives you automatic validation + docs. Flask is a solid alternative if you want something more minimal.

## 1. Install and set up

```bash
pip install "fastapi[standard]"
```

## 2. A complete CRUD API

```python
# main.py
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel
from typing import Optional

app = FastAPI(title="Bookstore API")

# --- Schemas ---
class BookCreate(BaseModel):
    title: str
    author: str
    year: Optional[int] = None

class Book(BookCreate):
    id: int

# --- In-memory "database" (swap for a real DB later) ---
books: dict[int, Book] = {}
_next_id = 1

# --- Routes ---
@app.get("/books", response_model=list[Book])
def list_books():
    return list(books.values())

@app.get("/books/{book_id}", response_model=Book)
def get_boo

In [5]:
from langchain.messages import AIMessage, SystemMessage, HumanMessage

# Create an AI message manually (e.g., for conversation history)
ai_msg = AIMessage("I'd be happy to help you with that question!")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("Can you help me?"),
    ai_msg,  # Insert as if it came from the model
    HumanMessage("Great! What's 2+2?")
]

response = model.invoke(messages)
print(response.content)

2 + 2 = 4


In [ ]:
# not optimal Hard coded
from langchain.messages import AIMessage,HumanMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result

In [7]:
tool_message

ToolMessage(content='Sunny, 72°F', tool_call_id='call_123')

In [8]:
response

AIMessage(content="It's currently sunny in San Francisco with a temperature of 72°F (about 22°C). A beautiful day in the city! 🌞", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 96, 'total_tokens': 164, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 35, 'rejected_prediction_tokens': None, 'text_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None, 'video_tokens': 0}, 'cost': 6.6e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 6.6e-05, 'upstream_inference_prompt_cost': 2.112e-05, 'upstream_inference_completions_cost': 4.488e-05}}, 'model_provider': 'openai', 'model_name': 'deepseek/deepseek-v4.1-flash', 'system_fingerprint': None, 'id': 'gen-1789904157-6vSdagdJ2MatmW8yAwcD', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a

In [9]:
# Best way to use tools
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny."

agent = create_agent(model=model, tools=[get_weather])

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in Karachi?"}]}
)
result["messages"][-1].content

"The weather in Karachi is currently sunny. Let me know if you'd like the forecast for another city!"

In [10]:
# with bind_tools (for more control)
from langchain_core.messages import HumanMessage

model_with_tools = model.bind_tools([get_weather])

messages = [HumanMessage("What's the weather in Karachi?")]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_msg = get_weather.invoke(tool_call)  # returns a ToolMessage
    messages.append(tool_msg)

final = model_with_tools.invoke(messages)
final.content

'The weather in Karachi is sunny. ☀️'